# Hyperparameter Tuning

**WICHTIG**
Dieses Notebook soll nicht nocheinmal komplet durchlaufen werden lassen. Denn das Tuning dauerte 45 Minuten!

## 1. Bibliotheken Import

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, ConfusionMatrixDisplay, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from scipy.stats import loguniform, randint
import matplotlib.pyplot as plt
import json
from pathlib import Path
import joblib

## 2. Daten einlesen und train_validate_test split

In [7]:
train_f = np.load("../../Model_data/ML_Daten/features_train.npz")
X_train_feat = train_f["X"]
y_train_raw = train_f["y"]

test_f = np.load("../../Model_data/ML_Daten/features_test.npz")
X_test_feat = test_f["X"]
y_test_raw = test_f["y"]

#Splitten der Trainingsdaten in Trainings- und Validierungsset
X_train, X_val, y_train, y_val = train_test_split(
    X_train_feat, y_train_raw,
    test_size=0.15,
    stratify=y_train_raw,
    random_state=42
)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded   = le.transform(y_val)
y_test_encoded  = le.transform(y_test_raw)
class_names = le.classes_

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test_feat)

## 3. Hyperparameter-Tuning des besten Modells

**Nich nochmals ausführen, ausser das Json ging verloren!!**

In [3]:
param_dist = {
    "n_estimators":     randint(100, 800),
    "max_depth":        randint(5, 30),        # oder None erlauben
    "min_samples_leaf": randint(1, 20),
    "max_features":     ["sqrt", "log2", 0.3, 0.5],
    "class_weight":     ["balanced", "balanced_subsample"]
}

base_model = RandomForestClassifier(random_state=42)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average="macro")
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=80,
    scoring=scorer,
    cv=folds,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_f1 = random_search.fit(X_train_scaled, y_train_encoded)
search_params = random_search.best_params_
print("\nBeste Hyperparameter-Kombination:")
for param, value in search_params.items():
    print(f"{param}: {value:.4f}" if isinstance(value, float) else f"{param}: {value}")

#Daten Speichern
out_dir = Path("../../Model_data/ML_Daten")
out_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "best_params": random_search.best_params_,
    "best_cv_score_macro_f1": float(random_search.best_score_),
}
with open(out_dir / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("Gespeichert:", out_dir / "best_hyperparameters.json")

Fitting 5 folds for each of 80 candidates, totalling 400 fits

Beste Hyperparameter-Kombination:
class_weight: balanced_subsample
max_depth: 24
max_features: log2
min_samples_leaf: 3
n_estimators: 211
Gespeichert: ..\..\Model_data\ML_Daten\best_hyperparameters.json


### Daten aus dem Json herausholen

In [4]:
data_dir = Path("../../Model_data/ML_Daten")

with open(data_dir / "best_hyperparameters.json", encoding="utf-8") as f:
    hp = json.load(f)
best_params = hp["best_params"]
# optional, falls vorhanden:
cv_score = hp.get("best_cv_score_macro_f1")
print("Train:", X_train_scaled.shape, "Test:", X_test_scaled.shape)
print("Klassen:", list(class_names))
print("best_params geladen:", best_params)
if cv_score is not None:
    print("Bestes CV-Score (macro F1):", cv_score)

Train: (12570, 165) Test: (10273, 165)
Klassen: [np.str_('Auto'), np.str_('Laufen'), np.str_('Lift'), np.str_('Roundkick'), np.str_('Treppe'), np.str_('Velo'), np.str_('Zug')]
best_params geladen: {'class_weight': 'balanced_subsample', 'max_depth': 24, 'max_features': 'log2', 'min_samples_leaf': 3, 'n_estimators': 211}
Bestes CV-Score (macro F1): 0.9608415351824517


### Model in Joblib speichern

In [5]:
best_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1)

best_model.fit(X_train_scaled, y_train_encoded)

# Validierungsset
y_val_pred = best_model.predict(X_val_scaled)
print("\nValidation-Set Performance:")
print(classification_report(y_val_encoded, y_val_pred, target_names=class_names))

# Erst danach: Testset (einmalig!)
y_test_pred = best_model.predict(X_test_scaled)
print("\n=== FINALE TEST PERFORMANCE ===")
print(classification_report(y_test_encoded, y_test_pred, target_names=class_names))


artifact = {
    "model": best_model,
    "scaler": scaler,
    "label_encoder": le,
    "class_names": list(class_names),
    "best_params": best_params,
}
out_path = Path("../../Model_data/ML_Daten/final_model.joblib")
joblib.dump(artifact, out_path)
print("Gespeichert:", out_path)


Validation-Set Performance:
              precision    recall  f1-score   support

        Auto       0.99      1.00      0.99       625
      Laufen       0.97      0.99      0.98       564
        Lift       0.97      0.95      0.96        87
   Roundkick       0.91      1.00      0.95        51
      Treppe       0.97      0.82      0.89        88
        Velo       0.98      0.97      0.97       215
         Zug       0.99      0.98      0.99       589

    accuracy                           0.98      2219
   macro avg       0.97      0.96      0.96      2219
weighted avg       0.98      0.98      0.98      2219


=== FINALE TEST PERFORMANCE ===
              precision    recall  f1-score   support

        Auto       0.93      0.15      0.26      1944
      Laufen       0.62      0.99      0.77      1529
        Lift       1.00      0.90      0.95       936
   Roundkick       0.99      0.84      0.91       132
      Treppe       0.67      0.49      0.57       263
        Velo    